In [15]:
import ee
import requests
import pandas as pd
import numpy as np

# ==============================================================================
# 1. CONSTANTES DE CONFIGURACIÓN
# ==============================================================================
GEE_PROJECT_ID = "gen-lang-client-0182377274"  # Reemplaza por el ID de tu proyecto de Google Cloud

# Rango temporal (Formato YYYY-MM-DD)
START_DATE = "2025-10-01"
END_DATE = "2026-03-31"

# Modo de localización: elige "POINT" (un punto) o "BBOX" (área cuadrada)
MODO = "BBOX"

# Si MODO = "POINT":
LAT = 37.8500
LON = -1.0500

# Si MODO = "BBOX": (min_lat, min_lon, max_lat, max_lon)
BBOX = (37.80, -1.30, 38.10, -0.95)

# Archivo de salida
OUTPUT_CSV = "datos_cultivos_completos.csv"


# ==============================================================================
# 2. AUTENTICACIÓN EN GOOGLE EARTH ENGINE
# ==============================================================================
print("1. Conectando con Google Earth Engine...")
try:
    ee.Initialize(project=GEE_PROJECT_ID)
except Exception:
    print("Iniciando autenticación interactiva de Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT_ID)

if MODO == "BBOX":
    min_lat, min_lon, max_lat, max_lon = BBOX
    geometria = ee.Geometry.BBox(min_lon, min_lat, max_lon, max_lat)
    query_lat = (min_lat + max_lat) / 2.0
    query_lon = (min_lon + max_lon) / 2.0
else:
    geometria = ee.Geometry.Point([LON, LAT])
    query_lat, query_lon = LAT, LON


# ==============================================================================
# 3. DESCARGA DE NASA POWER (Temperaturas, Precipitación, Radiación y Humedad)
# ==============================================================================
print("2. Descargando serie temporal de NASA POWER...")

parametros_power = [
    "T2M",                # Temperatura media a 2m (°C)
    "T2M_MIN",            # Temperatura mínima a 2m (°C)
    "T2M_MAX",            # Temperatura máxima a 2m (°C)
    "PRECTOTCORR",        # Precipitación total corregida (mm/día)
    "ALLSKY_SFC_SW_DWN",  # Radiación solar incidente (MJ/m²/día en comunidad AG)
    "GWETTOP",            # Humedad superficial del suelo NASA POWER (0 a 1)
    "GWETROOT",           # Humedad en la zona radicular NASA POWER (0 a 1)
]

power_url = "https://power.larc.nasa.gov/api/temporal/daily/point"
power_params = {
    "parameters": ",".join(parametros_power),
    "community": "AG",
    "longitude": query_lon,
    "latitude": query_lat,
    "start": START_DATE.replace("-", ""),
    "end": END_DATE.replace("-", ""),
    "format": "JSON",
}

resp = requests.get(power_url, params=power_params, timeout=30)
resp.raise_for_status()
power_json = resp.json()["properties"]["parameter"]

df_power = pd.DataFrame(power_json)
# Reemplazar posibles valores nulos de la NASA (-999.0) por NaN
df_power.replace(-999.0, np.nan, inplace=True)
df_power.index = pd.to_datetime(df_power.index, format="%Y%m%d")
df_power.index.name = "fecha"

# Renombrar columnas de NASA POWER para mayor claridad
df_power.rename(columns={
    "T2M": "temp_media_C",
    "T2M_MIN": "temp_min_C",
    "T2M_MAX": "temp_max_C",
    "PRECTOTCORR": "precipitacion_mm",
    "ALLSKY_SFC_SW_DWN": "radiacion_solar_MJ_m2",
    "GWETTOP": "humedad_suelo_power_idx",
    "GWETROOT": "humedad_raiz_power_idx"
}, inplace=True)


# ==============================================================================
# 4. DESCARGA DE NASA SMAP L4 EN GEE (Humedad y Temp. Suelo a 9 km)
# ==============================================================================
print("3. Descargando serie temporal de SMAP L4 desde Google Earth Engine...")

bandas_smap = [
    "sm_surface",       # Humedad superficial (0-5 cm) [m3/m3]
    "sm_rootzone",      # Humedad radicular (0-100 cm) [m3/m3]
    "soil_temp_layer1", # Temperatura del suelo capa 1 [K]
]

coleccion_smap = (
    ee.ImageCollection("NASA/SMAP/SPL4SMGP/008")
    .filterDate(START_DATE, END_DATE)
    .select(bandas_smap)
)

tabla_cruda = coleccion_smap.getRegion(geometry=geometria, scale=9000).getInfo()

cabeceras = tabla_cruda[0]
filas = tabla_cruda[1:]
df_smap_raw = pd.DataFrame(filas, columns=cabeceras)

df_smap_raw["fecha"] = pd.to_datetime(df_smap_raw["time"], unit="ms").dt.floor("D")
df_smap = df_smap_raw.groupby("fecha")[bandas_smap].mean()

# Pasar temperatura del suelo de Kelvin a Celsius y renombrar
df_smap["soil_temp_layer1"] = df_smap["soil_temp_layer1"] - 273.15
df_smap.rename(columns={
    "sm_surface": "humedad_suelo_smap_m3m3",
    "sm_rootzone": "humedad_raiz_smap_m3m3",
    "soil_temp_layer1": "temp_suelo_smap_C"
}, inplace=True)


# ==============================================================================
# 5. PROPIEDADES DEL SUELO EN GEE (pH, Carbono Orgánico y Textura - SoilGrids)
# ==============================================================================
print("4. Extrayendo pH, carbono orgánico y textura del suelo (SoilGrids 250m)...")

# Promediamos las 3 primeras capas superficiales/radiculares (0-5cm, 5-15cm, 15-30cm)
capas_0_30cm = ["0-5cm_mean", "5-15cm_mean", "15-30cm_mean"]

ph_img = ee.Image("projects/soilgrids-isric/phh2o_mean").select([f"phh2o_{c}" for c in capas_0_30cm]).reduce(ee.Reducer.mean()).rename("ph")
soc_img = ee.Image("projects/soilgrids-isric/soc_mean").select([f"soc_{c}" for c in capas_0_30cm]).reduce(ee.Reducer.mean()).rename("soc")
clay_img = ee.Image("projects/soilgrids-isric/clay_mean").select([f"clay_{c}" for c in capas_0_30cm]).reduce(ee.Reducer.mean()).rename("clay")
sand_img = ee.Image("projects/soilgrids-isric/sand_mean").select([f"sand_{c}" for c in capas_0_30cm]).reduce(ee.Reducer.mean()).rename("sand")
silt_img = ee.Image("projects/soilgrids-isric/silt_mean").select([f"silt_{c}" for c in capas_0_30cm]).reduce(ee.Reducer.mean()).rename("silt")

# Combinar todas las capas del suelo en una sola imagen para hacer 1 única petición a GEE
suelo_combinado = ee.Image.cat([ph_img, soc_img, clay_img, sand_img, silt_img])

# Extraer el valor medio del punto o área cuadrada a 250 metros de resolución
datos_suelo = suelo_combinado.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=geometria,
    scale=250,
    maxPixels=1e8
).getInfo()

# Función para clasificar la clase textural USDA según % de arcilla, limo y arena
def clasificar_textura_usda(arcilla, limo, arena):
    if any(v is None for v in [arcilla, limo, arena]):
        return "Desconocida"
    if arcilla >= 40 and limo < 40 and arena <= 45:
        return "Arcillosa (Clay)"
    elif arcilla >= 27 and arcilla < 40 and arena > 20 and arena <= 45:
        return "Franco-arcillosa (Clay Loam)"
    elif arcilla >= 7 and arcilla < 27 and limo >= 28 and limo < 50 and arena <= 52:
        return "Franca (Loam)"
    elif arena >= 52 and arcilla < 20:
        return "Franco-arenosa (Sandy Loam)"
    elif limo >= 50 and arcilla < 27:
        return "Franco-limosa (Silt Loam)"
    return "Otra / Mixta"

# Conversión de unidades nativas de ISRIC SoilGrids (vienen multiplicadas por 10)
ph_suelo = round(datos_suelo["ph"] / 10.0, 2) if datos_suelo.get("ph") else np.nan
carbono_org_g_kg = round(datos_suelo["soc"] / 10.0, 2) if datos_suelo.get("soc") else np.nan
arcilla_pct = round(datos_suelo["clay"] / 10.0, 1) if datos_suelo.get("clay") else np.nan
arena_pct = round(datos_suelo["sand"] / 10.0, 1) if datos_suelo.get("sand") else np.nan
limo_pct = round(datos_suelo["silt"] / 10.0, 1) if datos_suelo.get("silt") else np.nan
clase_textural = clasificar_textura_usda(arcilla_pct, limo_pct, arena_pct)


# ==============================================================================
# 6. UNIR TABLAS, AÑADIR VARIABLES DE SUELO Y EXPORTAR CSV
# ==============================================================================
print("5. Uniendo datos y guardando CSV...")

df_final = df_power.join(df_smap, how="inner").round(4)

# Añadir las propiedades edafológicas al DataFrame
df_final["suelo_ph_0_30cm"] = ph_suelo
df_final["suelo_carbono_organico_g_kg"] = carbono_org_g_kg
df_final["suelo_arcilla_pct"] = arcilla_pct
df_final["suelo_arena_pct"] = arena_pct
df_final["suelo_limo_pct"] = limo_pct
df_final["suelo_textura_clase"] = clase_textural

df_final.to_csv(OUTPUT_CSV)

print(f"\n¡Completado! Archivo guardado en: {OUTPUT_CSV} ({len(df_final)} días)")
print(f"Propiedades del suelo detectadas: pH={ph_suelo}, SOC={carbono_org_g_kg} g/kg, Textura={clase_textural} (Arcilla:{arcilla_pct}%, Arena:{arena_pct}%, Limo:{limo_pct}%)")
df_final.head()

1. Conectando con Google Earth Engine...
2. Descargando serie temporal de NASA POWER...


/var/folders/8h/ky58_y0164qd64b0h8110ctc0000gn/T/ipykernel_49883/832620575.py:87: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_power.rename(columns={


3. Descargando serie temporal de SMAP L4 desde Google Earth Engine...


/var/folders/8h/ky58_y0164qd64b0h8110ctc0000gn/T/ipykernel_49883/832620575.py:126: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_smap.rename(columns={


4. Extrayendo pH, carbono orgánico y textura del suelo (SoilGrids 250m)...
5. Uniendo datos y guardando CSV...

¡Completado! Archivo guardado en: datos_cultivos_completos.csv (181 días)
Propiedades del suelo detectadas: pH=7.88, SOC=17.4 g/kg, Textura=Franco-arcillosa (Clay Loam) (Arcilla:33.0%, Arena:23.5%, Limo:43.5%)


,temp_media_C,temp_min_C,temp_max_C,precipitacion_mm,radiacion_solar_MJ_m2,humedad_suelo_power_idx,humedad_raiz_power_idx,humedad_suelo_smap_m3m3,humedad_raiz_smap_m3m3,temp_suelo_smap_C,suelo_ph_0_30cm,suelo_carbono_organico_g_kg,suelo_arcilla_pct,suelo_arena_pct,suelo_limo_pct,suelo_textura_clase
fecha,,,,,,,,,,,,,,,,
2025-10-01,20.27,15.38,26.56,0.22,16.03,0.53,0.52,0.1698,0.1847,21.9125,7.88,17.4,33.0,23.5,43.5,Franco-arcillosa (Clay Loam)
2025-10-02,20.67,17.19,26.89,0.01,17.99,0.51,0.52,0.1568,0.1827,22.0015,7.88,17.4,33.0,23.5,43.5,Franco-arcillosa (Clay Loam)
2025-10-03,22.64,16.99,30.16,0.00,17.70,0.50,0.51,0.1481,0.1807,22.7391,7.88,17.4,33.0,23.5,43.5,Franco-arcillosa (Clay Loam)
2025-10-04,23.41,18.11,31.08,0.07,16.41,0.49,0.50,0.1357,0.1779,23.8328,7.88,17.4,33.0,23.5,43.5,Franco-arcillosa (Clay Loam)
2025-10-05,21.47,16.81,27.34,0.20,17.62,0.48,0.50,0.1238,0.1759,23.1691,7.88,17.4,33.0,23.5,43.5,Franco-arcillosa (Clay Loam)


In [14]:
print(df_final.keys())

Index(['temp_media_C', 'temp_min_C', 'temp_max_C', 'precipitacion_mm',
       'radiacion_solar_MJ_m2', 'humedad_suelo_power_idx',
       'humedad_raiz_power_idx', 'humedad_suelo_smap_m3m3',
       'humedad_raiz_smap_m3m3', 'temp_suelo_smap_C', 'suelo_ph_0_30cm',
       'suelo_carbono_organico_g_kg', 'suelo_arcilla_pct', 'suelo_arena_pct',
       'suelo_limo_pct', 'suelo_textura_clase'],
      dtype='str')
